# Detecção de outliers com Isolation Forest e registro no W&B

- Name: Renan Santos Mendes
- Email: renansantosmendes@gmail.com
- Course subject: `<COURSE_SUBJECT_PLACEHOLDER>`
- Program: `<PROGRAM_PLACEHOLDER>`

Versão didática do treino, alinhada ao pacote `outlier_detector` do
repositório. O código roda de cima para baixo, sem funções. O modelo é um
Isolation Forest que aprende o padrão de retornos, volume, retorno em
excesso ao Ibovespa e dias de divulgação de resultado. A avaliação é
walk-forward: o modelo é reajustado com dados do passado e pontua os dias
seguintes. No final, o modelo final é registrado no Weights & Biases (W&B).

## 1. Bibliotecas

Importa as bibliotecas usadas no notebook. Para instalar as dependências,
rode `uv sync --extra dev --extra notebook` na raiz do repositório.

In [ ]:
import os
import re
import tempfile
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import wandb
import yfinance as yf
from dotenv import load_dotenv
from sklearn.ensemble import IsolationForest

## 2. Classe do detector

Define a classe `IsolationForestDetector` neste notebook. Ela junta o
`IsolationForest` do scikit-learn com a lista de colunas usadas no treino:

- `fit` aprende o padrão dos dias de treino e guarda as colunas.
- `score_days` devolve, para cada dia, o `score` (quanto maior, mais
  atípico) e `is_atypical` (True quando o modelo classifica o dia como
  outlier).
- `save` grava um dicionário simples com o modelo e as configurações. Ele não
  grava a classe, então a API consegue carregar o arquivo mesmo sem este
  notebook.

In [ ]:
class IsolationForestDetector:
    """Score how isolated each trading day is from the training days.

    Parameters
    ----------
    contamination : float, optional
        Expected share of outliers in the training data. The default is 0.1.
    n_estimators : int, optional
        Number of isolation trees. The default is 300.
    random_seed : int, optional
        Seed that makes the trees reproducible. The default is 42.
    """

    def __init__(
        self,
        contamination: float = 0.1,
        n_estimators: int = 300,
        random_seed: int = 42,
    ) -> None:
        self.contamination = contamination
        self.n_estimators = n_estimators
        self.random_seed = random_seed
        self.forest = IsolationForest(
            n_estimators=n_estimators,
            contamination=contamination,
            random_state=random_seed,
        )
        self.feature_columns = []

    def fit(self, features: pd.DataFrame) -> "IsolationForestDetector":
        """Learn the typical shape of the rows in ``features``.

        Parameters
        ----------
        features : pd.DataFrame
            Numeric training rows without missing values.

        Returns
        -------
        IsolationForestDetector
            The same detector, now fitted.
        """
        self.feature_columns = list(features.columns)
        self.forest.fit(features)
        return self

    def score_days(self, features: pd.DataFrame) -> pd.DataFrame:
        """Score rows against the fitted model.

        Parameters
        ----------
        features : pd.DataFrame
            Rows with the same columns, in the same order, as the training
            rows.

        Returns
        -------
        pd.DataFrame
            Frame indexed like ``features`` with the columns ``score`` and
            ``is_atypical``.
        """
        return pd.DataFrame(
            {
                "score": -self.forest.score_samples(features),
                "is_atypical": self.forest.predict(features) == -1,
            },
            index=features.index,
        )

    def save(self, path: Path) -> None:
        """Write the fitted detector to a joblib file.

        Parameters
        ----------
        path : Path
            Destination file. Its parent directory must exist.
        """
        joblib.dump(
            {
                "forest": self.forest,
                "feature_columns": self.feature_columns,
                "contamination": self.contamination,
                "n_estimators": self.n_estimators,
                "random_seed": self.random_seed,
            },
            path,
        )

## 2. Parâmetros

Define o ativo, o benchmark e as configurações do treino. Para uma ação da
B3, use o sufixo `.SA` (por exemplo `PETR4.SA`). `CONTAMINATION` é a fração
esperada de outliers em cada treino. O primeiro treino usa cerca de 3 anos
de pregões (`TRAIN_MIN`), e o modelo é reajustado a cada cerca de 6 meses
(`STEP`).

In [ ]:
TICKER = "AAPL"
BENCHMARK = "^BVSP"
START = "2020-01-01"
CONTAMINATION = 0.1
N_ESTIMATORS = 300
TRAIN_MIN = 252 * 3
STEP = 126
RANDOM_SEED = 42
WANDB_PROJECT = "outlier-detector"

## 3. Preços do ativo

Baixa o histórico com preços ajustados e remove o fuso horário do índice.
Ficam só as colunas de fechamento e de volume, em `market_data`.

In [ ]:
market_history = yf.Ticker(TICKER).history(start=START, auto_adjust=True)
market_history.index = market_history.index.tz_localize(None)
market_data = market_history[["Close", "Volume"]].copy()

## 4. Features de retorno e volume

Cria as features a partir dos preços. Todas usam o dia atual ou dias
anteriores, nunca o futuro:

- **log_return**: retorno logarítmico diário.
- **absolute_log_return**: tamanho do movimento, sem o sinal.
- **previous_log_return**: retorno logarítmico do dia anterior.
- **volatility_20_days**: desvio padrão dos retornos dos últimos 20 dias.
- **relative_volume_20_days**: volume do dia dividido pela média dos últimos
  20 dias.

In [ ]:
market_data["log_return"] = np.log(market_data["Close"]).diff()
market_data["absolute_log_return"] = market_data["log_return"].abs()
market_data["previous_log_return"] = market_data["log_return"].shift(1)
market_data["volatility_20_days"] = (
    market_data["log_return"].rolling(20).std()
)
market_data["relative_volume_20_days"] = market_data["Volume"] / (
    market_data["Volume"].rolling(20).mean()
)

## 5. Retorno em excesso ao Ibovespa

Calcula `excess_log_return`: o retorno do ativo menos o retorno do
benchmark no mesmo dia. Se o benchmark não estiver disponível, o download
falha com erro, para que o modelo nunca seja treinado sem essa feature.

In [ ]:
benchmark_history = yf.download(
    BENCHMARK,
    start=START,
    auto_adjust=True,
    progress=False,
)["Close"].squeeze()
benchmark_history.index = pd.DatetimeIndex(
    benchmark_history.index
).tz_localize(None)
benchmark_log_returns = np.log(benchmark_history).diff()
market_data["excess_log_return"] = market_data["log_return"] - (
    benchmark_log_returns.reindex(market_data.index)
)

## 6. Dias de divulgação de resultado

Marca com `is_earnings_day = 1` os dias em que a empresa divulgou resultado.
Se as datas não estiverem disponíveis, o download falha com erro.

In [ ]:
earnings_history = yf.Ticker(TICKER).get_earnings_dates(limit=60)
earnings_dates = pd.DatetimeIndex(
    earnings_history.index
).tz_localize(None).normalize()
market_data["is_earnings_day"] = (
    market_data.index.normalize().isin(earnings_dates).astype(int)
)

## 7. Matriz de features

Escolhe as sete features do modelo, na mesma ordem do pacote, troca
infinitos por vazio e descarta os dias incompletos. O modelo só recebe as
colunas de `feature_columns`.

In [ ]:
feature_columns = [
    "log_return",
    "absolute_log_return",
    "previous_log_return",
    "volatility_20_days",
    "relative_volume_20_days",
    "excess_log_return",
    "is_earnings_day",
]
features = (
    market_data[feature_columns]
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
)
print(f"Dias com features completas: {len(features)}")

## 8. Walk-forward

Para cada bloco de `STEP` dias, um detector é treinado só com as linhas
anteriores ao bloco e pontua o próprio bloco, que ele ainda não viu. O
resultado de cada bloco é guardado em `walk_forward_parts` e depois
concatenado. Cada score vem de `score_days`: quanto maior, mais atípico o
dia, e `is_atypical` marca os outliers.

In [ ]:
if len(features) <= TRAIN_MIN:
    raise ValueError("Histórico insuficiente para o TRAIN_MIN definido.")

walk_forward_parts = []
for block_start in range(TRAIN_MIN, len(features), STEP):
    block_end = block_start + STEP
    detector = IsolationForestDetector(
        contamination=CONTAMINATION,
        n_estimators=N_ESTIMATORS,
        random_seed=RANDOM_SEED,
    ).fit(features.iloc[:block_start])
    walk_forward_parts.append(
        detector.score_days(features.iloc[block_start:block_end])
    )
walk_forward_results = pd.concat(walk_forward_parts)

## 9. Resultados

Mostra quantos dias foram pontuados fora da amostra de treino, quantos
foram marcados como outlier e os 15 outliers com maior score.

In [ ]:
outliers = walk_forward_results[walk_forward_results["is_atypical"]]
outlier_share = walk_forward_results["is_atypical"].mean()

print(f"Dias pontuados (out-of-sample): {len(walk_forward_results)}")
print(f"Outliers detectados: {len(outliers)} ({outlier_share:.2%})")
print("\nTop 15 outliers por score:")
print(
    outliers.sort_values("score", ascending=False)
    .head(15)
    .join(market_data[["Close", "log_return", "relative_volume_20_days"]])
)

## 10. Gráfico

Desenha o preço com os outliers em vermelho, e embaixo o score de anomalia.
A linha tracejada mostra onde começa o período avaliado pelo walk-forward.

In [ ]:
figure, (price_axis, score_axis) = plt.subplots(
    2,
    1,
    figsize=(14, 9),
    sharex=True,
    gridspec_kw={"height_ratios": [3, 1]},
)

price_axis.plot(
    market_data.index,
    market_data["Close"],
    lw=1,
    color="steelblue",
    label="Fechamento ajustado",
)
price_axis.scatter(
    outliers.index,
    market_data.loc[outliers.index, "Close"],
    color="red",
    s=35,
    zorder=3,
    label=f"Outliers walk-forward ({len(outliers)})",
)
price_axis.axvline(
    features.index[TRAIN_MIN],
    color="gray",
    ls="--",
    lw=1,
    label="Início do período avaliado",
)
price_axis.set_title(f"{TICKER} - outliers com Isolation Forest (walk-forward)")
price_axis.set_ylabel("Preço")
price_axis.legend()
price_axis.grid(alpha=0.3)

score_axis.plot(
    walk_forward_results.index,
    walk_forward_results["score"],
    lw=0.8,
    color="gray",
    label="Score de anomalia",
)
score_axis.scatter(
    outliers.index,
    outliers["score"],
    color="red",
    s=20,
    zorder=3,
)
score_axis.set_ylabel("Score")
score_axis.set_xlabel("Data")
score_axis.legend()
score_axis.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 11. Registro no Weights & Biases

A chave do W&B é lida do arquivo `.env` (`WANDB_API_KEY`), e nunca fica no
código. Com `WANDB_MODE=disabled` no `.env`, o notebook roda sem enviar nada
ao W&B. Esta célula só configura a conexão e confere a chave.

In [ ]:
load_dotenv()
run_mode = "disabled" if os.getenv("WANDB_MODE") == "disabled" else "online"
if run_mode == "online":
    if not os.getenv("WANDB_API_KEY"):
        raise RuntimeError("Defina WANDB_API_KEY no arquivo .env.")
    wandb.login(key=os.getenv("WANDB_API_KEY"), timeout=10)

### 11.1 Iniciar a run

A run guarda a configuração do treino. Ela é a mesma que o pacote registra,
para que as runs do notebook e do repositório sejam comparáveis.

In [ ]:
run_config = {
    "ticker": TICKER,
    "model_family": "isolation_forest",
    "feature_columns": feature_columns,
    "start": START,
    "benchmark": BENCHMARK,
    "contamination": CONTAMINATION,
    "n_estimators": N_ESTIMATORS,
    "random_seed": RANDOM_SEED,
    "train_min": TRAIN_MIN,
    "step": STEP,
}
run = wandb.init(project=WANDB_PROJECT, mode=run_mode, config=run_config)

### 11.2 Registrar as métricas

Registra quantas linhas foram usadas e qual fração dos dias avaliados foi
marcada como atípica no walk-forward.

In [ ]:
walk_forward_metrics = {
    "training_rows": len(features),
    "walk_forward_rows": len(walk_forward_results),
    "walk_forward_flagged_share": round(float(outlier_share), 4),
}
run.log(walk_forward_metrics)

### 11.3 Treinar o modelo final e registrar o artefato

O modelo final é ajustado com todas as features. Ele é salvo em um arquivo
temporário e enviado ao W&B como artefato `outlier-detector-<ticker>`, do
tipo `model`. A API usa esse artefato, na versão `latest`, quando inicia.

In [ ]:
final_detector = IsolationForestDetector(
    contamination=CONTAMINATION,
    n_estimators=N_ESTIMATORS,
    random_seed=RANDOM_SEED,
).fit(features)

ticker_slug = re.sub(r"[^\w.-]", "-", TICKER.lower())
model_directory = Path(tempfile.mkdtemp())
model_file = model_directory / f"{ticker_slug}.joblib"
final_detector.save(model_file)

artifact = wandb.Artifact(
    name=f"outlier-detector-{ticker_slug}",
    type="model",
    description=f"Isolation Forest outlier detector of {TICKER}.",
    metadata={
        **run_config,
        **walk_forward_metrics,
        "trained_at": pd.Timestamp.now(tz="UTC").isoformat(),
    },
)
artifact.add_file(str(model_file))
run.log_artifact(artifact)
print(f"Modelo registrado: outlier-detector-{ticker_slug}")

### 11.4 Encerrar a run

Encerra a run do W&B. Rode esta célula depois das anteriores, ou o
dashboard mostra a run como em andamento.

In [ ]:
run.finish()